<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 02. VGG

## Very Deep Convolutional Networks for Large-Scale Image Recognition

- **원 논문:** [Very Deep Convolutional Networks for Large-Scale Image Recognition](https://arxiv.org/abs/1409.1556)
- **저자 / 발표:** Karen Simonyan, Andrew Zisserman · ICLR 2015 (2014)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 연속 3×3 convolution의 receptive field와 2/2/3 block을 재현한다.

**축소하거나 생략한 부분:** 16/19 weight layers와 ImageNet 대신 7-conv, 16×16 합성 도형을 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1: stride-1 3×3 convolution | Task 1 | receptive field assert |
| §2.2 Table 1: configurations | Task 2 | 2/2/3 conv blocks |
| §2.3: small filters | Task 1 | 3×3 stack/7×7 parameters |
| §3: classification framework | Task 3 | loss/accuracy |

## 핵심 아이디어와 수식

$$R_L=1+\sum_{\ell=1}^{L}(k_\ell-1)$$

**기호 해설:** R_L은 누적 receptive field, L은 층 수, k_l은 각 convolution kernel 크기다.

**코드 대응:** Task 1이 receptive field와 parameter 수를 비교하고 Task 2가 2/2/3 block을 만든다.

**입력과 출력 shape:** 입력 [B,1,16,16] → block별 8,4,2 해상도 → GAP [B,16] → [B,4].

**포트폴리오 구현 범위:** VGG-16의 13 conv 대신 7 conv를 쓰지만 작은 kernel을 깊게 쌓는 원칙은 동일하다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1: stride-1 3×3 convolution
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** receptive field assert

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class VGGConfig:
    block_depths: tuple = (2, 2, 3)
    block_channels: tuple = (8, 12, 16)
    num_classes: int = 4
    learning_rate: float = 0.01
    steps: int = 12


def receptive_field(kernels):
    return 1 + sum(kernel - 1 for kernel in kernels)


def convolution_parameters(in_channels, out_channels, kernel):
    return in_channels * out_channels * kernel * kernel


config = VGGConfig()
stacked = 3 * convolution_parameters(64, 64, 3)
single = convolution_parameters(64, 64, 7)
assert receptive_field([3, 3, 3]) == receptive_field([7]) == 7
assert stacked < single

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2 Table 1: configurations
- **노트북에서 구현할 부분:** Task 2
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 2/2/3 conv blocks

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def make_vgg_block(in_channels, out_channels, depth):
    layers = []
    for index in range(depth):
        current_in = in_channels if index == 0 else out_channels
        layers.append(nn.Conv2d(current_in, out_channels, 3, padding=1))
        layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(kernel_size=2))
    return nn.Sequential(*layers)


class VGGMini(nn.Module):
    def __init__(self, config):
        super().__init__()
        blocks = []
        in_channels = 1
        for depth, out_channels in zip(
            config.block_depths,
            config.block_channels,
        ):
            blocks.append(make_vgg_block(in_channels, out_channels, depth))
            in_channels = out_channels
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(in_channels, config.num_classes)

    def forward_features(self, inputs):
        return self.features(inputs)

    def forward(self, inputs):
        features = self.forward_features(inputs)
        return self.classifier(features.mean(dim=(2, 3)))


model = VGGMini(config).to(DEVICE)
conv_count = sum(isinstance(layer, nn.Conv2d) for layer in model.modules())
assert conv_count == 7
assert model(images[:2]).shape == (2, 4)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3: classification framework
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** loss/accuracy

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def classification_loss(logits, targets):
    return F.cross_entropy(logits, targets)


def train_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = classification_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_classifier(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    step_loss = train_step(model, optimizer, x_train, y_train)
    losses.append(step_loss)

accuracy = evaluate_classifier(
    model,
    images[test_idx],
    labels[test_idx],
)
assert min(losses[3:]) < losses[0]
assert 0.0 <= accuracy <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("VGG mini: accuracy={:.1%}".format(accuracy))
plt.tight_layout()
plt.show()
print(f"loss={losses[0]:.3f} -> {losses[-1]:.3f}")

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 16/19 weight layers와 ImageNet 대신 7-conv, 16×16 합성 도형을 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.